# Inference Demo — High-IoU Prediction

Runs the trained Prithvi-100M model on a single patch selected for visual quality.  
Saves the result to `results/linkedin_prediction_demo.png` in Drive.

**Environment**: Google Colab (CPU or GPU, any tier)  
**Runtime**: < 3 minutes

In [ ]:
# ── Install dependencies (restarts kernel once if needed) ─────────────────────
import subprocess, sys, os

try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    print(f'Installing ({e})...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==2.0.2', '--force-reinstall'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
# ── Imports & setup ───────────────────────────────────────────────────────────
import warnings
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from terratorch.registry import BACKBONE_REGISTRY
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
import subprocess

BASE     = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
CKPT     = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
RESULTS  = BASE / 'results'

# Copy patches to /content/ for fast access
LOCAL = Path('/content/patches')
LOCAL.mkdir(exist_ok=True)
if not (LOCAL / 'images').exists():
    print('Copying patches to /content/ ...')
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/images'),     str(LOCAL)], check=True)
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks_dnbr'), str(LOCAL)], check=True)
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks'),      str(LOCAL)], check=True)
    print('Done.')

IMG_DIR   = LOCAL / 'images'
DNBR_DIR  = LOCAL / 'masks_dnbr'
FIRMS_DIR = LOCAL / 'masks'

img_paths   = sorted(IMG_DIR.glob('*.npy'))
dnbr_paths  = sorted(DNBR_DIR.glob('*.npy'))
firms_paths = sorted(FIRMS_DIR.glob('*.npy'))
print(f'Patches: {len(img_paths)}')
assert CKPT.exists(), f'Checkpoint not found: {CKPT}'
print(f'Checkpoint: OK ({CKPT.stat().st_size/1e6:.0f} MB)')

In [ ]:
# ── Select patch: same filter as linkedin_figure_v8 ──────────────────────────
print('Scanning patches for best visual quality...')
candidates = []
for i, (ip, fp, dp) in enumerate(tqdm(zip(img_paths, firms_paths, dnbr_paths))):
    firms      = np.load(fp, mmap_mode='r')
    dnbr       = np.load(dp, mmap_mode='r')
    raw        = np.load(ip, mmap_mode='r').astype(np.float32)
    firms_rate = float(firms.mean())
    dnbr_rate  = float(dnbr.mean())
    contrast   = float(raw[1].std())
    unburned   = raw[1][dnbr == 0]
    if len(unburned) < 500:
        continue
    unburned_green = float(unburned.mean())
    if (firms_rate == 0.0
            and 0.28 < dnbr_rate < 0.75
            and unburned_green > 700
            and contrast > 350):
        candidates.append((i, contrast, dnbr_rate))

candidates.sort(key=lambda x: x[1], reverse=True)
PATCH_IDX = candidates[8][0]   # same index used for linkedin_figure_v8
print(f'Selected patch index: {PATCH_IDX}')
print(f'  File : {img_paths[PATCH_IDX].name}')
print(f'  dNBR : {candidates[8][2]:.1%}')

In [ ]:
# ── Model architecture (identical to notebook 07) ─────────────────────────────
class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side
    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]
        B, L, D = x.shape
        return x.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        feats = self.backbone(x.unsqueeze(2))
        return self.decoder(self.neck(feats))


print('Loading Prithvi backbone...')
backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
for p in backbone.parameters():
    p.requires_grad = False

model = PrithviSegModel(backbone).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print('Model loaded.')

In [ ]:
# ── Inference on selected patch ───────────────────────────────────────────────
BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE     = 224
T            = (256 - IMG_SIZE) // 2
THRESHOLD    = 0.5

raw_img  = np.load(img_paths[PATCH_IDX]).astype(np.float32)    # (11,256,256)
raw_dnbr = np.load(dnbr_paths[PATCH_IDX]).astype(np.float32)   # (256,256)
raw_firms= np.load(firms_paths[PATCH_IDX]).astype(np.float32)

# Preprocess
img_norm = raw_img[BAND_IDX] / 10000.0
img_norm = (img_norm - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
img_crop = img_norm[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
dnbr_crop= raw_dnbr[T:T+IMG_SIZE, T:T+IMG_SIZE]
firms_crop= raw_firms[T:T+IMG_SIZE, T:T+IMG_SIZE]

tensor = torch.from_numpy(img_crop).unsqueeze(0).to(DEVICE)    # (1,6,224,224)

with torch.no_grad():
    prob = model(tensor).sigmoid().squeeze().cpu().numpy()      # (224,224)

pred = (prob > THRESHOLD).astype(np.float32)

# Compute IoU
tp = (pred * dnbr_crop).sum()
fp = (pred * (1 - dnbr_crop)).sum()
fn = ((1 - pred) * dnbr_crop).sum()
iou = tp / (tp + fp + fn + 1e-6)
recall    = tp / (tp + fn + 1e-6)
precision = tp / (tp + fp + 1e-6)
print(f'IoU      : {iou:.3f}')
print(f'Recall   : {recall:.3f}')
print(f'Precision: {precision:.3f}')

In [ ]:
# ── Generate figure ───────────────────────────────────────────────────────────
def norm_band(b, p=1):
    v = b[b > 0]
    lo, hi = np.percentile(v, [p, 100-p]) if len(v) else (0, 1)
    return np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)

raw_crop = raw_img[:, T:T+IMG_SIZE, T:T+IMG_SIZE] / 10000.0
rgb = np.dstack([norm_band(raw_crop[2]), norm_band(raw_crop[1]), norm_band(raw_crop[0])])

# Severity map (post-fire NBR)
nir  = raw_img[4, T:T+IMG_SIZE, T:T+IMG_SIZE].astype(np.float32)
swir = raw_img[6, T:T+IMG_SIZE, T:T+IMG_SIZE].astype(np.float32)
nbr  = (nir - swir) / (nir + swir + 1e-6)
sev_colors = ['#7a0000', '#cc0000', '#ff6600', '#ffcc00', '#78c800']
sev_bounds = [-1.0, -0.05, 0.10, 0.25, 0.40, 1.0]
sev_cmap   = LinearSegmentedColormap.from_list('sev', sev_colors, N=256)
sev_norm   = BoundaryNorm(sev_bounds, sev_cmap.N)

# Error map
err = np.zeros((*pred.shape, 3))
err[(pred==1)&(dnbr_crop==1)] = [0.0, 0.82, 0.0]
err[(pred==1)&(dnbr_crop==0)] = [1.0, 0.50, 0.0]
err[(pred==0)&(dnbr_crop==1)] = [0.90, 0.10, 0.10]

import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(22, 7), facecolor='white')
outer = gridspec.GridSpec(2, 1, figure=fig,
                          height_ratios=[1, 0.10],
                          hspace=0.08, left=0.02, right=0.98,
                          top=0.88, bottom=0.02)

fig.suptitle(
    f'Prithvi-100M Burn Scar Prediction  —  IoU = {iou:.3f}  |  Recall = {recall:.3f}  |  Precision = {precision:.3f}',
    fontsize=14, fontweight='bold', color='#111111', y=0.97
)

inner = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[0], wspace=0.04)

panels = [
    ('Sentinel-2 RGB',      '#111111', rgb,        None,           None),
    ('FIRMS Labels',        '#d4500a', rgb,        firms_crop,     [1.0, 0.42, 0.13, 0.85]),
    ('dNBR Ground Truth',   '#c03030', rgb,        dnbr_crop,      [0.95, 0.25, 0.25, 0.85]),
    ('Burn Severity (NBR)', '#555555', nbr,        None,           None),
    ('Prediction + Errors', '#111111', err,        None,           None),
]

for col_i, (title, tc, data, mask, color) in enumerate(panels):
    ax = fig.add_subplot(inner[col_i])
    ax.axis('off')
    ax.set_title(title, color=tc, fontsize=11, fontweight='bold', pad=6)

    if col_i == 3:
        ax.imshow(data, cmap=sev_cmap, norm=sev_norm)
        sev_patches = [
            mpatches.Patch(color='#7a0000', label='High'),
            mpatches.Patch(color='#cc0000', label='Moderate'),
            mpatches.Patch(color='#ff6600', label='Low'),
            mpatches.Patch(color='#78c800', label='Unburned'),
        ]
        ax.legend(handles=sev_patches, loc='lower left', fontsize=7,
                  framealpha=0.80, edgecolor='#cccccc', handlelength=1.2, borderpad=0.5)
    elif col_i == 4:
        ax.imshow(rgb)
        ax.imshow(np.dstack([err, np.where(err.sum(axis=2)>0, 0.85, 0.0)]))
        iou_txt = f'IoU = {iou:.3f}'
        ax.text(5, 15, iou_txt, color='white', fontsize=10, fontweight='bold',
                bbox={'fc': '#111111', 'alpha': 0.7, 'pad': 3})
        tp_p = mpatches.Patch(color=[0,.82,0], label='True Positive')
        fp_p = mpatches.Patch(color=[1,.50,0], label='False Positive')
        fn_p = mpatches.Patch(color=[.9,.10,.10], label='False Negative')
        ax.legend(handles=[tp_p, fp_p, fn_p], loc='lower left', fontsize=7,
                  framealpha=0.80, edgecolor='#cccccc', handlelength=1.2, borderpad=0.5)
    else:
        ax.imshow(data)
        if mask is not None:
            ov = np.zeros((*mask.shape, 4))
            ov[mask > 0] = color
            ax.imshow(ov)
            pct = mask.mean() * 100
            ax.text(5, 215, f'{pct:.1f}% labeled', color='#111111', fontsize=9,
                    bbox={'fc': 'white', 'alpha': 0.75, 'pad': 3})

# Thin caption bar at bottom
ax_cap = fig.add_subplot(outer[1])
ax_cap.axis('off')
ax_cap.text(0.5, 0.5,
    'Model: Prithvi-EO-1.0-100M (IBM/NASA) + FPN decoder  |  Labels: dNBR > 0.10  |  Region: Corrientes, Argentina 2022',
    ha='center', va='center', fontsize=9, color='#666666',
    transform=ax_cap.transAxes)

OUT = RESULTS / 'linkedin_prediction_demo.png'
plt.savefig(OUT, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {OUT}')